In [ ]:
import os, duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/kaggle/input/music-listening-data-500k-users"  # <- adjust if your folder name differs
DB_PATH  = os.path.join(DATA_DIR, "music.duckdb")

con = duckdb.connect(DB_PATH, read_only=True)
print("Connected to:", DB_PATH)
sql = con.sql

In [ ]:
overview = sql("""
SELECT 
  (SELECT COUNT(*) FROM users)                    AS users,
  (SELECT COUNT(*) FROM user_top_artists)        AS artist_rows,
  (SELECT COUNT(*) FROM user_top_tracks)         AS track_rows,
  (SELECT COUNT(*) FROM user_top_albums)         AS album_rows
""").df()

overview

In [ ]:
top_artists = sql("""
SELECT artist_name, COUNT(DISTINCT user_id) AS fans
FROM user_top_artists
GROUP BY artist_name
ORDER BY fans DESC
LIMIT 25
""").df()

top_artists.tail(10)  # peek

plt.figure(figsize=(7,8))
top_artists.sort_values("fans").plot.barh(x="artist_name", y="fans", legend=False)
plt.title("Global Top Artists (by unique users)")
plt.xlabel("Unique users")
plt.tight_layout()
plt.show()

In [ ]:
country_top = sql("""
WITH fans AS (
  SELECT u.country, a.artist_name, COUNT(*) AS fans
  FROM user_top_artists a
  JOIN users u USING (user_id)
  WHERE u.country <> ''
  GROUP BY 1,2
),
ranked AS (
  SELECT country, artist_name, fans,
         ROW_NUMBER() OVER (PARTITION BY country ORDER BY fans DESC) AS rk
  FROM fans
)
SELECT country, artist_name, fans
FROM ranked
WHERE rk = 1
ORDER BY fans DESC
LIMIT 25
""").df()

country_top